# Lecture 6 — Class Exercise
## Part-to-Whole: Hierarchical Visualization

> **Push to:** `week06/lecture06_exercise.ipynb`

**Rules:**
1. Use `px` first, then customise with `update_traces` / `update_layout`
2. Colour encodes a meaningful category — not decoration
3. Insight title names the specific finding
4. Consider: would a bar chart be clearer? If yes, use the bar chart

---


In [9]:
import pandas as pd
import plotly.express as px
import numpy as np

# Dataset: Global Energy Mix by Country and Source
df = pd.read_csv('..data/global_energy_mix.csv')

# Source type mapping
source_category = {
    'Coal': 'Fossil', 'Oil': 'Fossil', 'Natural Gas': 'Fossil',
    'Nuclear': 'Low-carbon', 'Hydro': 'Low-carbon',
    'Wind': 'Renewable', 'Solar': 'Renewable', 'Other Renewables': 'Renewable'
}
df['Source_Type'] = df['Source'].map(source_category)

print(f"Loaded: {len(df)} rows")
print(df.head(10))

FileNotFoundError: [Errno 2] No such file or directory: '..data/global_energy_mix.csv'

## Task 1 — Treemap: fossil fuel dependency by country


In [ ]:
# Task 1 — Treemap: fossil fuel dependency by country

# Filter to fossil sources only
fossil = df[df['Source_Type'] == 'Fossil'].copy()

# CVD-safe palette for Coal / Oil / Natural Gas
fossil_colors = {
    'Coal':        '#E63946',
    'Oil':         '#F4A261',
    'Natural Gas': '#457B9D'
}

fig = px.treemap(
    fossil,
    path=['Region', 'Country', 'Source'],
    values='TWh',
    color='Source',
    color_discrete_map={
        **fossil_colors,
        '(?)': '#CCCCCC'   # grey for parent nodes
    },
    title='Asia-Pacific dominates global fossil fuel consumption, led by coal'
)

fig.update_traces(
    texttemplate='%{label}<br>%{value:.0f} TWh',
    textfont=dict(family='Arial', size=11),
    marker=dict(line=dict(width=1, color='white'))
)

fig.update_layout(
    paper_bgcolor='white',
    font=dict(family='Arial'),
    title=dict(font=dict(family='Arial', size=15, color='#222222')),
    margin=dict(t=50, l=10, r=10, b=10)
)

fig.show()

## Task 2 — Sunburst: tipping behaviour by day and meal time


In [ ]:
# Task 2 — Sunburst: total bill by day → time → smoker

tips = px.data.tips()

# Aggregate total bill per group
tips_agg = (
    tips.groupby(['day', 'time', 'smoker'])['total_bill']
    .sum()
    .reset_index()
)

# CVD-safe blue/orange for smoker status
smoker_colors = {
    'No':  '#457B9D',   # blue  = non-smoker
    'Yes': '#F4A261',   # orange = smoker
    '(?)': '#CCCCCC'    # grey for parent nodes
}

fig = px.sunburst(
    tips_agg,
    path=['day', 'time', 'smoker'],
    values='total_bill',
    color='smoker',
    color_discrete_map=smoker_colors,
    title='Saturday dinner drives the most restaurant spending — non-smokers dominate'
)

fig.update_traces(
    texttemplate='%{label}<br>%{percentParent:.0%}',
    textfont=dict(family='Arial', size=11),
    insidetextorientation='horizontal'
)

fig.update_layout(
    paper_bgcolor='white',
    font=dict(family='Arial'),
    title=dict(font=dict(family='Arial', size=15, color='#222222')),
    margin=dict(t=50, l=10, r=10, b=10)
)

fig.show()

## Task 3 — Treemap vs bar: low-carbon energy by country


In [ ]:
# Task 3 — Treemap vs bar chart: low-carbon TWh by country

# Filter and aggregate
lowcarbon = (
    df[df['Source_Type'] == 'Low-carbon']
    .groupby('Country')['TWh']
    .sum()
    .reset_index()
    .sort_values('TWh', ascending=False)
)
lowcarbon['All'] = 'Low-carbon'  # dummy root node

# --- Treemap ---
fig_tree = px.treemap(
    lowcarbon,
    path=['All', 'Country'],
    values='TWh',
    color='TWh',
    color_continuous_scale='Blues',
    title='Low-carbon energy by country (treemap)'
)

fig_tree.update_traces(
    texttemplate='%{label}<br>%{value:.0f} TWh',
    textfont=dict(family='Arial', size=11),
    marker=dict(line=dict(width=1, color='white'))
)

fig_tree.update_layout(
    paper_bgcolor='white',
    font=dict(family='Arial'),
    title=dict(font=dict(family='Arial', size=14, color='#222222')),
    coloraxis_showscale=False,
    margin=dict(t=50, l=10, r=10, b=10)
)
fig_tree.show()

# --- Bar chart ---
fig_bar = px.bar(
    lowcarbon.sort_values('TWh', ascending=True),
    x='TWh',
    y='Country',
    orientation='h',
    text='TWh',
    color_discrete_sequence=['#457B9D'],
    title='China leads all countries in low-carbon energy generation'
)

fig_bar.update_traces(
    texttemplate='%{text:.0f}',
    textposition='outside',
    marker_color='#457B9D'
)

fig_bar.update_layout(
    paper_bgcolor='white',
    plot_bgcolor='white',
    font=dict(family='Arial'),
    title=dict(font=dict(family='Arial', size=15, color='#222222')),
    xaxis=dict(
        title='TWh',
        showgrid=True, gridcolor='#F0F0F0',
        showline=False
    ),
    yaxis=dict(showgrid=False, showline=False),
    margin=dict(l=120, r=80)
)
fig_bar.show()

### Which chart is clearer?

The **bar chart** is clearer for comparing low-carbon energy by country. It allows precise reading of values along a common axis, making it easy to rank countries and see exact differences. The treemap shows proportional size well but makes it harder to compare countries with similar values — especially smaller ones. When the task is ranking or comparing magnitudes, a sorted bar chart wins.